In [4]:
from dataclasses import dataclass
from enum import Enum, auto
class Action(Enum):
    F = auto()
    B = auto()


@dataclass
class MicroBatch:
    action:Action
    mb_id:int

    def __repr__(self):
        return f"[{self.action.name},{self.mb_id}]"



class MicroBatchSchedular:
    def __init__(
        self, 
        world_size:int, 
        world_rank:int,
        num_microbatches:int = 8,
        max_activation_size:int = 4,

        ):
        self.world_size = world_size
        self.world_rank = world_rank
        self.num_microbatches = num_microbatches
        self.max_activation_size = max_activation_size
        if num_microbatches < max_activation_size:
            raise AttributeError(f"num_microbatches({num_microbatches}) can not be lesser than max_activation_size({max_activation_size})")
        
    def build(self):
        schedulers = list()
        forward_lists = list()
        backward_lists = list()



        for i in range(self.world_size):
            schedulers.append(list())
            forward_lists.append(list())
            backward_lists.append(list())

        forward_lists[0] = list(range(self.num_microbatches))

        backward_round = [False] * self.world_size
        activation_count = [0] * self.world_size
        while True:
            no_action = 0
            for i in range(self.world_size):
                if backward_round[i]:
                    if len(backward_lists[i]) != 0:
                        back_id = backward_lists[i].pop(0)
                        schedulers[i].append(MicroBatch(Action.B,back_id))
                        activation_count[i] -= 1
                        if i != 0:
                            backward_lists[i-1].append(back_id)
                        if len(forward_lists[i]) != 0:
                            backward_round[i] = False
                    elif len(forward_lists[i]) != 0: #not likely to come in.
                        backward_round[i] = False
                    else:
                        no_action += 1

                else:
                    if len(forward_lists[i]) != 0 and activation_count[i] < self.max_activation_size:
                        for_id = forward_lists[i].pop(0)
                        schedulers[i].append(MicroBatch(Action.F, for_id))
                        activation_count[i] += 1
                        if i == self.world_size - 1:
                            backward_lists[i].append(for_id)
                        else:
                            forward_lists[i+1].append(for_id)
                        if len(backward_lists[i]) != 0:
                            backward_round[i] = True
                    elif len(backward_lists[i]) != 0:
                        back_id = backward_lists[i].pop(0)
                        schedulers[i].append(MicroBatch(Action.B, back_id))
                        activation_count[i] -= 1
                        if i != 0:
                            backward_lists[i-1].append(back_id)
                        
                    else:
                        no_action += 1
            print(activation_count)
            if no_action == self.world_size:
                break
        return schedulers

In [5]:
dd = MicroBatchSchedular(4, 0).build()

[1, 1, 1, 1]
[2, 2, 2, 0]
[3, 3, 3, 1]
[4, 4, 2, 0]
[4, 3, 3, 1]
[3, 3, 2, 0]
[4, 4, 1, 1]
[4, 3, 2, 0]
[3, 2, 1, 1]
[4, 1, 1, 0]
[3, 2, 2, 0]
[4, 3, 1, 1]
[3, 2, 2, 0]
[4, 3, 3, 1]
[3, 3, 2, 0]
[3, 2, 1, 1]
[2, 1, 1, 0]
[1, 1, 0, 0]
[1, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]


In [6]:
for i in dd:
    print(i)

[[F,0], [F,1], [F,2], [F,3], [B,0], [F,4], [B,1], [F,5], [B,2], [F,6], [B,3], [F,7], [B,4], [B,5], [B,6], [B,7]]
[[F,0], [F,1], [F,2], [F,3], [B,0], [F,4], [B,1], [B,2], [B,3], [F,5], [F,6], [B,4], [F,7], [B,5], [B,6], [B,7]]
[[F,0], [F,1], [F,2], [B,0], [F,3], [B,1], [B,2], [F,4], [B,3], [F,5], [B,4], [F,6], [F,7], [B,5], [B,6], [B,7]]
[[F,0], [B,0], [F,1], [B,1], [F,2], [B,2], [F,3], [B,3], [F,4], [B,4], [F,5], [B,5], [F,6], [B,6], [F,7], [B,7]]
